# SAM2 Scratch Fine-tuning

# 1. Connect Google Drive

In [ ]:
"""
    Connect Google Drive for Colab training.

    Purpose:
        - Read the prepared SAM2 dataset from Drive.
        - Save checkpoints, configs, logs, and training curves back to Drive.
        - Keep Colab local disk as temporary fast runtime storage only.
"""
from google.colab import drive

# Mount Drive once at /content/drive.
drive.mount('/content/drive')

# 2. Settings

In [ ]:
"""
    Global SAM2 fine-tuning configuration.

    Dataset:
        DATASET_NAME        : folder/zip name uploaded to Drive.
        LOCAL_DATASET_ROOT  : local Colab copy used during training.

    Model:
        PRETRAINED_CKPT_NAME: official SAM2.1 Hiera Base Plus checkpoint.
        RESOLUTION          : training image size. Current data is 1024x1024 patches.
        VISION_LR           : 0.0 freezes the image encoder for safer small-data tuning.

    Training:
        EPOCHS              : number of full passes over the expanded train list.
        BATCH_SIZE          : keep 1 for Colab T4/limited VRAM.
        DATASET_MULTIPLIER  : repeats the dataset inside each epoch.
        BASE_LR             : learning rate for trainable SAM2 modules.

    Output:
        LOCAL_EXP_DIR       : temporary Colab run directory.
        DRIVE_RUN_DIR       : permanent Drive backup for checkpoint and logs.
"""
from pathlib import Path
from datetime import datetime

"""Drive folders"""
DRIVE_PROJECT = Path('/content/drive/MyDrive/Surface-Scratch-Detection').resolve()
DRIVE_DATA_DIR = DRIVE_PROJECT / 'data'
DRIVE_MODEL_DIR = DRIVE_PROJECT / 'models' / 'sam2'

"""Dataset folders"""
DATASET_NAME = 'scratch_sam2_format'
LOCAL_DATASET_ROOT = Path('/content') / DATASET_NAME

"""SAM2 repo and run folders"""
SAM2_REPO = Path('/content/sam2')
RUN_NAME = 'scratch_sam2_bplus_1024_' + datetime.now().strftime('%Y%m%d_%H%M%S')
LOCAL_EXP_DIR = Path('/content/sam2_runs') / RUN_NAME
DRIVE_RUN_DIR = DRIVE_MODEL_DIR / RUN_NAME

"""Training hyperparameters"""
RESOLUTION = 1024
EPOCHS = 30
BATCH_SIZE = 1
NUM_WORKERS = 2
NUM_GPUS = 1
DATASET_MULTIPLIER = 4
BASE_LR = '1.0e-5'
VISION_LR = '0.0'

"""Official pretrained SAM2.1 Base Plus checkpoint"""
PRETRAINED_CKPT_NAME = 'sam2.1_hiera_base_plus.pt'
PRETRAINED_CKPT_URL = (
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/'
    'sam2.1_hiera_base_plus.pt'
)

print('Drive project:', DRIVE_PROJECT)
print('Local dataset:', LOCAL_DATASET_ROOT)
print('Run name:', RUN_NAME)


# 3. Install SAM2

In [ ]:
"""
    Install official facebookresearch/sam2 inside Colab.

    Flow:
        1. Clone SAM2 repo if /content/sam2 does not exist.
        2. Install the repo in editable mode with dev extras.
        3. Put SAM2_REPO into PYTHONPATH so training imports resolve correctly.
"""
import os

if not SAM2_REPO.exists():
    !git clone https://github.com/facebookresearch/sam2.git "{SAM2_REPO}"

%cd "{SAM2_REPO}"
!python -m pip install -q -U pip
!python -m pip install -q -e ".[dev]"

# Make `training.*` and `sam2.*` imports visible to later cells.
os.environ['PYTHONPATH'] = str(SAM2_REPO)
print('SAM2 repo:', SAM2_REPO)


# 4. Download SAM2.1 Base Plus Checkpoint

In [ ]:
"""
    Download the pretrained SAM2.1 checkpoint used as fine-tune initialization.

    Input:
        PRETRAINED_CKPT_URL : official public checkpoint URL.

    Output:
        /content/sam2/checkpoints/sam2.1_hiera_base_plus.pt

    Safety check:
        After download, the file must exist and have non-zero size.
"""
checkpoint_dir = SAM2_REPO / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
pretrained_ckpt = checkpoint_dir / PRETRAINED_CKPT_NAME

if not pretrained_ckpt.is_file():
    !wget -q -O "{pretrained_ckpt}" "{PRETRAINED_CKPT_URL}"

if not pretrained_ckpt.is_file() or pretrained_ckpt.stat().st_size == 0:
    raise RuntimeError(f'Checkpoint download failed: {pretrained_ckpt}')

print('Pretrained checkpoint:', pretrained_ckpt)
print('Size MB:', round(pretrained_ckpt.stat().st_size / 1024 / 1024, 2))


# 5. Prepare Dataset From Drive

In [ ]:
"""
    Locate and copy the SAM2-format scratch dataset into Colab local disk.

    Accepted Drive layouts:
        - /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_sam2_format/
        - /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_sam2_format.zip
        - /content/drive/MyDrive/scratch_sam2_format/
        - /content/drive/MyDrive/scratch_sam2_format.zip
"""
import shutil
import zipfile

candidate_dirs = [
    DRIVE_DATA_DIR / DATASET_NAME,
    DRIVE_PROJECT / DATASET_NAME,
    Path('/content/drive/MyDrive') / DATASET_NAME,
    Path('/content') / DATASET_NAME,
]

candidate_zips = [
    DRIVE_DATA_DIR / f'{DATASET_NAME}.zip',
    DRIVE_PROJECT / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive') / f'{DATASET_NAME}.zip',
    Path('/content') / f'{DATASET_NAME}.zip',
]

def find_dataset_source() -> Path:
    """
        Find a dataset folder or extract a dataset zip.

        Returns:
            Path to a folder with SAM2 layout:
                JPEGImages/<split>/<sample_id>/00000.png
                Annotations/<split>/<sample_id>/00000.png
                ImageSets/train.txt, valid.txt, test.txt
    """
    for path in candidate_dirs:
        if path.is_dir():
            return path

    for zip_path in candidate_zips:
        if not zip_path.is_file():
            continue
        extract_root = Path('/content/datasets')
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {zip_path} -> {extract_root}')
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(extract_root)
        extracted = extract_root / DATASET_NAME
        if extracted.is_dir():
            return extracted

    raise FileNotFoundError(
        'Dataset not found. Upload scratch_sam2_format/ or '
        'scratch_sam2_format.zip to Drive.'
    )

source_dataset = find_dataset_source()
print('Source dataset:', source_dataset)

# Replace an old local copy only when it is different from the selected source.
if LOCAL_DATASET_ROOT.exists() and LOCAL_DATASET_ROOT.resolve() != source_dataset.resolve():
    shutil.rmtree(LOCAL_DATASET_ROOT)

if LOCAL_DATASET_ROOT.resolve() != source_dataset.resolve():
    print(f'Copying dataset to local runtime: {LOCAL_DATASET_ROOT}')
    shutil.copytree(source_dataset, LOCAL_DATASET_ROOT, symlinks=False)
else:
    print('Dataset already local')

print('Dataset root:', LOCAL_DATASET_ROOT)


# 6. Validate SAM2 Dataset

In [ ]:
"""
    Validate the SAM2 dataset before training.

    Expected structure:
        JPEGImages/<split>/<sample_id>/00000.png
        Annotations/<split>/<sample_id>/00000.png
        ImageSets/<split>.txt

    Checks:
        - train/valid/test file lists exist and are non-empty.
        - every listed image and mask exists.
        - image size == RESOLUTION x RESOLUTION.
        - mask size == RESOLUTION x RESOLUTION.
        - mask values are binary {0, 1}.
        - each sample contains exactly one non-empty instance mask.
"""
from PIL import Image, UnidentifiedImageError
import numpy as np

jpeg_root = LOCAL_DATASET_ROOT / 'JPEGImages'
ann_root = LOCAL_DATASET_ROOT / 'Annotations'
imagesets_root = LOCAL_DATASET_ROOT / 'ImageSets'

required = [
    jpeg_root,
    ann_root,
    imagesets_root / 'train.txt',
    imagesets_root / 'valid.txt',
    imagesets_root / 'test.txt',
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing SAM2 dataset paths:
' + '
'.join(str(p) for p in missing))

def read_list(split: str) -> list[str]:
    """Read sample ids for one split from ImageSets/<split>.txt."""
    path = imagesets_root / f'{split}.txt'
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

def validate_split(split: str) -> None:
    """
        Validate one SAM2 split.

        Args:
            split: one of train, valid, test

        Raises:
            FileNotFoundError if an expected image/mask is missing.
            RuntimeError if image format, size, or mask values are invalid.
    """
    names = read_list(split)
    if not names:
        raise RuntimeError(f'Empty file list: {split}')

    mask_values = set()
    checked = 0
    for name in names:
        image_path = jpeg_root / name / '00000.png'
        mask_path = ann_root / name / '00000.png'
        if not image_path.is_file():
            raise FileNotFoundError(f'Missing image: {image_path}')
        if not mask_path.is_file():
            raise FileNotFoundError(f'Missing mask: {mask_path}')

        try:
            with Image.open(image_path) as image:
                image_size = image.size
        except UnidentifiedImageError as exc:
            size = image_path.stat().st_size if image_path.exists() else 0
            raise RuntimeError(
                f'Cannot read image as JPEG: {image_path}
'
                f'File size: {size} bytes. This usually means the dataset was '
                'uploaded from a symlink build. Recreate it with '
                '`--image-mode copy`, zip/upload again, then rerun this cell.'
            ) from exc

        try:
            with Image.open(mask_path) as mask:
                mask_arr = np.array(mask)
        except UnidentifiedImageError as exc:
            size = mask_path.stat().st_size if mask_path.exists() else 0
            raise RuntimeError(
                f'Cannot read mask as PNG: {mask_path}
'
                f'File size: {size} bytes.'
            ) from exc

        if image_size != (RESOLUTION, RESOLUTION):
            raise RuntimeError(f'Unexpected image size {image_size}: {image_path}')
        if mask_arr.shape[:2] != (RESOLUTION, RESOLUTION):
            raise RuntimeError(f'Unexpected mask size {mask_arr.shape}: {mask_path}')

        unique_values = set(int(v) for v in np.unique(mask_arr))
        mask_values |= unique_values
        if not unique_values.issubset({0, 1}):
            raise RuntimeError(f'Unexpected mask values {unique_values}: {mask_path}')
        if 1 not in unique_values:
            raise RuntimeError(f'Empty object mask: {mask_path}')
        checked += 1

    print(f'{split}: samples={len(names)} checked={checked} mask_values={sorted(mask_values)}')

for split in ['train', 'valid', 'test']:
    validate_split(split)


# 7. Create SAM2 Fine-tune Config

In [ ]:
"""
    Create a SAM2 training config for scratch fine-tuning.

    Base config:
        sam2.1_hiera_b+_MOSE_finetune.yaml

    Converted task:
        MOSE-style video object segmentation -> one-frame scratch instance segmentation.

    Key replacements:
        resolution      : 1024 for current patch dataset.
        num_frames      : 1 because each sample is a static image.
        max_num_objects : 1 because dataset was split to one mask per sample.
        vision_lr       : 0.0 to freeze image encoder.
        file_list_txt   : ImageSets/train.txt.
        checkpoint_path : official SAM2.1 Base Plus pretrained checkpoint.
"""
base_config = SAM2_REPO / 'sam2' / 'configs' / 'sam2.1_training' / 'sam2.1_hiera_b+_MOSE_finetune.yaml'
train_config = SAM2_REPO / 'sam2' / 'configs' / 'sam2.1_training' / 'scratch_sam2_finetune.yaml'

text = base_config.read_text(encoding='utf-8')
replacements = {
    '  resolution: 1024': f'  resolution: {RESOLUTION}',
    '  train_batch_size: 1': f'  train_batch_size: {BATCH_SIZE}',
    '  num_train_workers: 10': f'  num_train_workers: {NUM_WORKERS}',
    '  num_frames: 8': '  num_frames: 1',
    '  max_num_objects: 3': '  max_num_objects: 1',
    '  base_lr: 5.0e-6': f'  base_lr: {BASE_LR}',
    '  vision_lr: 3.0e-06': f'  vision_lr: {VISION_LR}',
    '  num_epochs: 40': f'  num_epochs: {EPOCHS}',
    '  img_folder: null # PATH to MOSE JPEGImages folder': f'  img_folder: {jpeg_root} # scratch JPEGImages folder',
    '  gt_folder: null  # PATH to MOSE Annotations folder': f'  gt_folder: {ann_root}  # scratch Annotations folder',
    '  file_list_txt: training/assets/MOSE_sample_train_list.txt # Optional PATH to filelist containing a subset of videos to be used for training': f'  file_list_txt: {imagesets_root / "train.txt"} # scratch train list',
    '  multiplier: 2': f'  multiplier: {DATASET_MULTIPLIER}',
    '        checkpoint_path: ./checkpoints/sam2.1_hiera_base_plus.pt # PATH to SAM 2.1 checkpoint': f'        checkpoint_path: {pretrained_ckpt} # PATH to SAM 2.1 checkpoint',
    '  experiment_log_dir: null # Path to log directory, defaults to ./sam2_logs/${config_name}': f'  experiment_log_dir: {LOCAL_EXP_DIR} # scratch run log directory',
    '    num_frames_to_correct_for_train: 2': '    num_frames_to_correct_for_train: 1',
    '    rand_frames_to_correct_for_train: True': '    rand_frames_to_correct_for_train: False',
    '    num_init_cond_frames_for_train: 2': '    num_init_cond_frames_for_train: 1',
    '    rand_init_cond_frames_for_train: True': '    rand_init_cond_frames_for_train: False',
}

# String replacement is used because the official training config is Hydra/OmegaConf YAML.
for old, new in replacements.items():
    if old not in text:
        print('Warning: replacement anchor not found:', old)
    text = text.replace(old, new)

train_config.write_text(text, encoding='utf-8')
print('Config written:', train_config)
print('Experiment dir:', LOCAL_EXP_DIR)


# 8. Smoke Test Config

In [ ]:
"""
    Smoke-test the generated Hydra config before launching training.

    Purpose:
        - Catch YAML/Hydra path mistakes early.
        - Confirm scratch-specific values were actually written.
        - Avoid waiting minutes before discovering a config error.
"""
import os
import sys
from pathlib import Path

os.chdir(SAM2_REPO)
if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))

from hydra import compose, initialize_config_module
from hydra.core.global_hydra import GlobalHydra
from training.utils.train_utils import register_omegaconf_resolvers

try:
    register_omegaconf_resolvers()
except Exception as exc:
    print('OmegaConf resolvers already registered or skipped:', exc)

# Colab can rerun cells; Hydra must be cleared before re-initialization.
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

with initialize_config_module('sam2', version_base='1.2'):
    cfg = compose(config_name='configs/sam2.1_training/scratch_sam2_finetune.yaml')

print('CWD:', Path.cwd())
print('resolution:', cfg.scratch.resolution)
print('num_frames:', cfg.scratch.num_frames)
print('max_num_objects:', cfg.scratch.max_num_objects)
print('img_folder:', cfg.dataset.img_folder)
print('gt_folder:', cfg.dataset.gt_folder)
print('file_list_txt:', cfg.dataset.file_list_txt)
print('experiment_log_dir:', cfg.launcher.experiment_log_dir)


# 9. Train SAM2

In [ ]:
"""
    Train SAM2 with the generated scratch config.

    What happens inside training/train.py:
        - Loads the pretrained SAM2.1 Base Plus checkpoint.
        - Builds one-frame scratch samples from JPEGImages/Annotations.
        - Simulates prompt-based mask correction during training.
        - Writes checkpoints, config snapshots, logs, and train_stats.json.

    Note:
        The first epochs can be slow because SAM2 is much heavier than U-Net/YOLO.
        If the cell is stopped early, check LOCAL_EXP_DIR/checkpoints for the latest
        checkpoint written by the trainer.
"""
%cd "{SAM2_REPO}"

!python training/train.py   -c configs/sam2.1_training/scratch_sam2_finetune.yaml   --use-cluster 0   --num-gpus {NUM_GPUS}


# 10. Save Epoch Training History


In [ ]:
"""
    Convert SAM2 training logs into easy-to-compare result files.

    Input:
        LOCAL_EXP_DIR/logs/train_stats.json

    Outputs:
        LOCAL_EXP_DIR/results.json
        LOCAL_EXP_DIR/results.csv
        LOCAL_EXP_DIR/train_stats.jsonl
        LOCAL_EXP_DIR/training_curves.png       (if matplotlib works)

    Drive backup:
        The same files are copied to DRIVE_RUN_DIR so they survive Colab runtime reset.

    Why this cell exists:
        SAM2 logs are JSONL metric rows. This cell reshapes them into epoch history
        similar to U-Net training outputs, making later comparison easier.
"""
import csv
import json
import math
import shutil
from pathlib import Path

train_stats_path = LOCAL_EXP_DIR / 'logs' / 'train_stats.json'
if not train_stats_path.is_file():
    raise FileNotFoundError(
        f'SAM2 train stats not found: {train_stats_path}
'
        'Run the training cell first, then run this cell.'
    )

def is_numeric(value) -> bool:
    """Return True for finite scalar metrics that can be saved to CSV/JSON."""
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))

def clean_metric_name(name: str) -> str:
    """Convert SAM2 metric keys such as Losses/train_all_loss into CSV-safe names."""
    return (
        name.replace('/', '_')
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace('.', '_')
    )

"""Read one JSON metric row per epoch from SAM2 train_stats.json."""
rows = []
with train_stats_path.open('r', encoding='utf-8') as file:
    for line_index, line in enumerate(file, start=1):
        line = line.strip()
        if not line:
            continue
        raw = json.loads(line)
        raw_epoch = int(raw.get('Trainer/epoch', len(rows)))
        row = {
            'epoch': raw_epoch + 1,
            'raw_epoch': raw_epoch,
        }

        for key, value in raw.items():
            if key in {'Trainer/epoch', 'Trainer/where'}:
                continue
            if is_numeric(value):
                row[clean_metric_name(key)] = float(value)

        # Pick the first train loss key as the main curve for quick inspection.
        loss_candidates = [
            key for key in row
            if key.startswith('Losses_train_') and key.endswith('_loss')
        ]
        if loss_candidates:
            row['train_loss'] = row[sorted(loss_candidates)[0]]

        rows.append(row)

if not rows:
    raise RuntimeError(f'No epoch rows found in {train_stats_path}')

"""Build a stable column order: summary columns first, all other metrics after."""
preferred_keys = ['epoch', 'train_loss', 'raw_epoch', 'Trainer_steps_train']
all_keys = []
for key in preferred_keys:
    if any(key in row for row in rows):
        all_keys.append(key)
for row in rows:
    for key in row:
        if key not in all_keys:
            all_keys.append(key)

history = {
    key: [row.get(key) for row in rows]
    for key in all_keys
}

"""Save local metric artifacts."""
LOCAL_EXP_DIR.mkdir(parents=True, exist_ok=True)
results_json = LOCAL_EXP_DIR / 'results.json'
results_csv = LOCAL_EXP_DIR / 'results.csv'
raw_jsonl = LOCAL_EXP_DIR / 'train_stats.jsonl'

results_json.write_text(json.dumps(history, indent=2, ensure_ascii=True), encoding='utf-8')
with results_csv.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_keys)
    writer.writeheader()
    for row in rows:
        writer.writerow({key: row.get(key) for key in all_keys})
shutil.copy2(train_stats_path, raw_jsonl)

"""Copy metric artifacts to Drive immediately."""
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(results_json, DRIVE_RUN_DIR / 'results.json')
shutil.copy2(results_csv, DRIVE_RUN_DIR / 'results.csv')
shutil.copy2(raw_jsonl, DRIVE_RUN_DIR / 'train_stats.jsonl')

"""Optional quick loss curve for visual training review."""
curve_path = None
if 'train_loss' in history and any(value is not None for value in history['train_loss']):
    try:
        import matplotlib.pyplot as plt

        curve_path = LOCAL_EXP_DIR / 'training_curves.png'
        plt.figure(figsize=(8, 5))
        plt.plot(history['epoch'], history['train_loss'], marker='o', label='train_loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('SAM2 Training History')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(curve_path, dpi=160)
        plt.show()
        shutil.copy2(curve_path, DRIVE_RUN_DIR / 'training_curves.png')
    except Exception as exc:
        print('Plot skipped:', exc)

print('Epoch rows:', len(rows))
print('Saved local:', results_json)
print('Saved local:', results_csv)
print('Saved Drive:', DRIVE_RUN_DIR / 'results.json')
print('Saved Drive:', DRIVE_RUN_DIR / 'results.csv')
if curve_path is not None:
    print('Saved Drive:', DRIVE_RUN_DIR / 'training_curves.png')


# 11. Save Checkpoint And Logs To Drive

In [ ]:
"""
    Save final checkpoint, config, logs, and TensorBoard files to Drive.

    Required input:
        LOCAL_EXP_DIR/checkpoints/checkpoint.pt

    Drive output:
        DRIVE_RUN_DIR/checkpoint.pt
        DRIVE_RUN_DIR/scratch_sam2_finetune.yaml
        DRIVE_RUN_DIR/config.yaml
        DRIVE_RUN_DIR/config_resolved.yaml
        DRIVE_RUN_DIR/logs/
        DRIVE_RUN_DIR/tensorboard/

    Use this cell after training finishes or after stopping at a useful checkpoint.
"""
import shutil

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

local_checkpoint = LOCAL_EXP_DIR / 'checkpoints' / 'checkpoint.pt'
if not local_checkpoint.is_file():
    checkpoints = sorted((LOCAL_EXP_DIR / 'checkpoints').glob('*.pt')) if (LOCAL_EXP_DIR / 'checkpoints').is_dir() else []
    raise FileNotFoundError(
        f'Final checkpoint not found: {local_checkpoint}
'
        f'Available checkpoints: {checkpoints}'
    )

"""Copy the final model and the exact config used for this run."""
shutil.copy2(local_checkpoint, DRIVE_RUN_DIR / 'checkpoint.pt')
shutil.copy2(train_config, DRIVE_RUN_DIR / 'scratch_sam2_finetune.yaml')

"""Copy optional run metadata when official SAM2 trainer produced it."""
if (LOCAL_EXP_DIR / 'config.yaml').is_file():
    shutil.copy2(LOCAL_EXP_DIR / 'config.yaml', DRIVE_RUN_DIR / 'config.yaml')
if (LOCAL_EXP_DIR / 'config_resolved.yaml').is_file():
    shutil.copy2(LOCAL_EXP_DIR / 'config_resolved.yaml', DRIVE_RUN_DIR / 'config_resolved.yaml')
if (LOCAL_EXP_DIR / 'logs').is_dir():
    shutil.copytree(LOCAL_EXP_DIR / 'logs', DRIVE_RUN_DIR / 'logs', dirs_exist_ok=True)
if (LOCAL_EXP_DIR / 'tensorboard').is_dir():
    shutil.copytree(LOCAL_EXP_DIR / 'tensorboard', DRIVE_RUN_DIR / 'tensorboard', dirs_exist_ok=True)

print('Saved to Drive:', DRIVE_RUN_DIR)
print('Checkpoint:', DRIVE_RUN_DIR / 'checkpoint.pt')
print('Checkpoint size MB:', round((DRIVE_RUN_DIR / 'checkpoint.pt').stat().st_size / 1024 / 1024, 2))


# 12. Optional: Download Checkpoint From Colab

In [ ]:
"""
    Optional Colab browser download for the final checkpoint.

    Normal workflow:
        The checkpoint is already saved to Drive by the previous cell.

    Use this cell when:
        You also want a direct browser download from the Colab session.
"""
from google.colab import files

checkpoint_on_drive = DRIVE_RUN_DIR / 'checkpoint.pt'
if checkpoint_on_drive.is_file():
    files.download(str(checkpoint_on_drive))
else:
    print('Checkpoint is not available yet:', checkpoint_on_drive)
